In [58]:
import cv2
import pandas as pd
import pytesseract
import os

In [28]:
# Path to the Tesseract executable 
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
video_path = 'Streams/HLE vs T1 Game1.mp4'
cap = cv2.VideoCapture(video_path)
fps = round(cap.get(cv2.CAP_PROP_FPS)) 

extracted_data = []
frame_count = 0

# Tesseract configuration: 
# Only recognize digits, colon, 'K', and period.
custom_config = r'--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789:K.'

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_count % fps == 0:
        if frame is None:
            continue
            
        # Cropping the top HUD area from the frame
        y_start, y_end = 0, 150
        x_start, x_end = 380, 1540
        top_hud = frame[y_start:y_end, x_start:x_end]
        
        # Gray-scaling and thresholding to create a binary image for better OCR performance
        gray_hud = cv2.cvtColor(top_hud, cv2.COLOR_BGR2GRAY)
        _, binary_hud = cv2.threshold(gray_hud, 150, 255, cv2.THRESH_BINARY)
        
        # Cropping smaller ROIs (Region of Interest) from the binary HUD
        
        # Time
        time_crop = binary_hud[55:120, 520:640] 
        
        # Blue team gold 
        blue_gold_crop = binary_hud[15:75, 350:450]
        
        # Red team gold 
        red_gold_crop = binary_hud[15:75, 700:800]

        # Blue team towers
        blue_tower_crop = binary_hud[20:60, 310:350]

        # Red team towers
        red_tower_crop = binary_hud[20:60, 810:840]
                
        
        # HUD visibility check (if there are not enough white pixels, the HUD is not visible)
        if cv2.countNonZero(time_crop) < 50 or cv2.countNonZero(blue_gold_crop) < 50 or cv2.countNonZero(red_gold_crop) < 50:
            time_str = ""
            blue_gold_str = ""
            red_gold_str = ""
            blue_tower_str = ""
            red_tower_str = ""
        else:
            # OCR processing on the cropped images
            time_str = pytesseract.image_to_string(time_crop, config=custom_config).strip()
            blue_gold_str = pytesseract.image_to_string(blue_gold_crop, config=custom_config).strip()
            red_gold_str = pytesseract.image_to_string(red_gold_crop, config=custom_config).strip()
            blue_tower_str = pytesseract.image_to_string(blue_tower_crop, config=custom_config).strip()
            red_tower_str = pytesseract.image_to_string(red_tower_crop, config=custom_config).strip()
        
        # Visualization of the cropped areas for debugging purposes
        cv2.imshow('Time', time_crop)
        cv2.imshow('Blue Gold', blue_gold_crop)
        cv2.imshow('Red Gold', red_gold_crop)
        cv2.imshow('Blue Towers', blue_tower_crop)
        cv2.imshow('Red Towers', red_tower_crop)
        
        print(f"[{frame_count//fps}s] Time: {time_str} | Blue Gold: {blue_gold_str} | Red Gold: {red_gold_str} | Blue Towers: {blue_tower_str} | Red Towers: {red_tower_str}")
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Processing interrupted.")
            break
        
        # Saving the extracted data into a list of dictionaries
        extracted_data.append({
            'video_sec': frame_count // fps,
            'match_time': time_str,
            'blue_total_gold': blue_gold_str,
            'red_total_gold': red_gold_str,
            'blue_towers': blue_tower_str,
            'red_towers': red_tower_str
        })
        
    frame_count += 1

cap.release()
cv2.destroyAllWindows()

df = pd.DataFrame(extracted_data)
print(df.head())
df.to_csv('extracted_data.csv', index=False)

[0s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[1s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[2s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[3s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[4s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[5s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[6s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[7s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[8s] Time:  | Blue Gold: 4 | Red Gold: : | Blue Towers:  | Red Towers: 
[9s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[10s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[11s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[12s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[13s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[14s] Time:  |

In [ ]:
# Processing the lower HUD area for KDA and CS extraction
video_path = 'Streams/HLE vs T1 Game1.mp4'
cap = cv2.VideoCapture(video_path)
fps = round(cap.get(cv2.CAP_PROP_FPS))

extracted_lower_data = []
frame_count = 0

custom_config_kda = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789/'
custom_config_cs = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789'

def get_binary(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    scaled = cv2.resize(gray, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)
    _, binary = cv2.threshold(scaled, 160, 255, cv2.THRESH_BINARY_INV)
    return binary

def do_ocr(binary_image, config):
    return pytesseract.image_to_string(binary_image, config=config).strip()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_count % fps == 0:
        if frame is None:
            continue
            
        height, width = frame.shape[:2]
        y_start = int(height * 0.75)
        y_end = int(height)
        
        # Cropping the lower HUD area for KDA and CS extraction
        blue_kda_crop = frame[y_start:y_end, 790:875]
        blue_cs_crop  = frame[y_start:y_end, 860:900]
        red_cs_crop   = frame[y_start:y_end, 1020:1060]
        red_kda_crop  = frame[y_start:y_end, 1045:1130]
        
        # Binary conversion for OCR
        b_kda_bin = get_binary(blue_kda_crop)
        b_cs_bin  = get_binary(blue_cs_crop)
        r_cs_bin  = get_binary(red_cs_crop)
        r_kda_bin = get_binary(red_kda_crop)
        
        # Visualization of the cropped areas for debugging purposes
        cv2.imshow('Blue KDA', blue_kda_crop)
        cv2.imshow('Blue CS', blue_cs_crop)
        cv2.imshow('Red CS', red_cs_crop)
        cv2.imshow('Red KDA', red_kda_crop)
        
        # HUD visibility check
        # Due to THRESH_BINARY_INV, white numbers on a black background will be black (0), and the dark background will be white (255).
        if cv2.mean(blue_kda_crop)[0] > 100:  # If too bright, the HUD table is not present
            blue_kda_str, blue_cs_str, red_cs_str, red_kda_str = "", "", "", ""
        else:
            # OCR extraction
            blue_kda_str = " ".join(do_ocr(b_kda_bin, custom_config_kda).split())
            blue_cs_str = " ".join(do_ocr(b_cs_bin, custom_config_cs).split())
            red_cs_str = " ".join(do_ocr(r_cs_bin, custom_config_cs).split())
            red_kda_str = " ".join(do_ocr(r_kda_bin, custom_config_kda).split())
            
        
        print(f"[{frame_count//fps}s] B-KDA: {blue_kda_str} | B-CS: {blue_cs_str} | R-CS: {red_cs_str} | R-KDA: {red_kda_str}")
        
        extracted_lower_data.append({
            'video_sec': frame_count // fps,
            'blue_kda': blue_kda_str,
            'blue_cs': blue_cs_str,
            'red_cs': red_cs_str,
            'red_kda': red_kda_str
        })
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Processing interrupted.")
            break
            
    frame_count += 1

cap.release()
cv2.destroyAllWindows()

df_lower = pd.DataFrame(extracted_lower_data)
print(df_lower.head())
df_lower.to_csv('hle_vs_t1_game1_lower_data.csv', index=False)


[0s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[1s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[2s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[3s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[4s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[5s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[6s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[7s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[8s] B-KDA: 2 | B-CS:  | R-CS:  | R-KDA: 
[9s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[10s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[11s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[12s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[13s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[14s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[15s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[16s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[17s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[18s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[19s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[20s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[21s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[22s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[23s] B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
[

In [81]:
import os
import cv2
import pandas as pd

# ============================================================
# VIDEO
# ============================================================

video_path = 'Streams/HLE vs T1 Game1.mp4'

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30

print(f"FPS: {fps}")

# ============================================================
# BEÁLLÍTÁSOK
# ============================================================

GAME_START_OFFSET = 0
MAX_SECONDS = 40  
ICONS_DIR = 'Custom_Icons'  

# ============================================================
# CHAMPION IDENTIFICATION (TEMPLATE MATCHING)
# ============================================================

def identify_champions(frame, icons_dir=ICONS_DIR):
    """Kivágja a 10 portrét, MEGMUTATJA ŐKET, majd párosítja az ikonokkal."""
    
    # KIVÁGÁSI KOORDINÁTÁK FINOMHANGOLÁSA (frame[y_start:y_end, x_start:x_end])
    portraits = {
        'blue_top': frame[140:190, 65:115],
        'blue_jungle': frame[255:305, 65:115],
        'blue_mid': frame[370:420, 65:115],
        'blue_adc': frame[470:520, 65:115],
        'blue_support': frame[585:635, 65:115],
        
        'red_top': frame[140:190, 1805:1855],
        'red_jungle': frame[255:305, 1805:1855],
        'red_mid': frame[370:420, 1805:1855],
        'red_adc': frame[470:520, 1805:1855],
        'red_support': frame[585:635, 1805:1855]
    }
    
    # --- VIZUÁLIS ELLENŐRZÉS BLOKK (IMSHOW) ---
    print("\n[DEBUG] Portrék megjelenítése ellenőrzéshez...")
    for role, crop in portraits.items():
        if crop.size > 0:
            cv2.imshow(role, crop)
            
    print("Nyomj meg bármilyen gombot az ablakok bezárásához és a folytatáshoz! (Ha a vágás rossz, zárd be a scriptet és írd át a koordinátákat)")
    cv2.waitKey(0) # Végtelen ideig vár egy gombnyomásra
    cv2.destroyAllWindows()
    # ------------------------------------------
    
    reference_icons = {}
    if os.path.exists(icons_dir):
        for filename in os.listdir(icons_dir):
            if (filename.endswith('.png') or filename.endswith('.jpg')) and filename.count('_') == 1:
                champ_name = filename.split('.')[0] 
                icon_path = os.path.join(icons_dir, filename)
                img = cv2.imread(icon_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    reference_icons[champ_name] = img
    else:
        print(f"FIGYELMEZTETÉS: A '{icons_dir}' mappa nem létezik!")

    identified_team = {}
    
    for role, crop in portraits.items():
        best_match = "Unknown"
        if crop.size > 0 and reference_icons:
            crop_gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            crop_h, crop_w = crop_gray.shape
            
            best_score = -1.0
            for champ_name, ref_img in reference_icons.items():
                ref_resized = cv2.resize(ref_img, (crop_w, crop_h), interpolation=cv2.INTER_AREA)
                result = cv2.matchTemplate(crop_gray, ref_resized, cv2.TM_CCOEFF_NORMED)
                _, max_val, _, _ = cv2.minMaxLoc(result)
                
                if max_val > best_score:
                    best_score = max_val
                    best_match = champ_name
                    
            if best_score < 0.3: 
                best_match = "Unknown"
                
        identified_team[role] = best_match
        
    return identified_team

# ============================================================
# DATA & MAIN LOOP
# ============================================================

extracted_data = []
processed_seconds = 0

current_teams = {
    'blue_top': 'Unknown', 'blue_jungle': 'Unknown', 'blue_mid': 'Unknown', 'blue_adc': 'Unknown', 'blue_support': 'Unknown',
    'red_top': 'Unknown', 'red_jungle': 'Unknown', 'red_mid': 'Unknown', 'red_adc': 'Unknown', 'red_support': 'Unknown'
}
champions_identified = False

while True:
    target_frame = int(processed_seconds * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
    ret, frame = cap.read()

    if not ret or (MAX_SECONDS is not None and processed_seconds >= MAX_SECONDS):
        break

    match_seconds = max(0, processed_seconds - GAME_START_OFFSET)
    minutes = match_seconds // 60
    seconds = match_seconds % 60
    match_time = f"{minutes:02d}:{seconds:02d}"

    # ========================================================
    # CHAMPION IDENTIFICATION 
    # ========================================================
    if match_seconds == 30 and not champions_identified:
        print(f"\n[{match_time}] --- Hősök azonosítása... ---")
        current_teams = identify_champions(frame)
        champions_identified = True
        print("Azonosítás eredménye:")
        for role, champ in current_teams.items():
            print(f"  {role}: {champ}")
        print("------------------------------------\n")

    # ========================================================
    # SAVE & STATUS
    # ========================================================
    
    if processed_seconds % 30 == 0 and processed_seconds > 5:
        print(f"[{match_time}] Feldolgozás alatt...")

    extracted_data.append({
        'video_sec': processed_seconds,
        'match_time': match_time,
        
        'blue_top_champ': current_teams['blue_top'],
        'blue_jungle_champ': current_teams['blue_jungle'],
        'blue_mid_champ': current_teams['blue_mid'],
        'blue_adc_champ': current_teams['blue_adc'],
        'blue_support_champ': current_teams['blue_support'],
        
        'red_top_champ': current_teams['red_top'],
        'red_jungle_champ': current_teams['red_jungle'],
        'red_mid_champ': current_teams['red_mid'],
        'red_adc_champ': current_teams['red_adc'],
        'red_support_champ': current_teams['red_support']
    })

    processed_seconds += 1

# ============================================================
# RELEASE & SAVE CSV
# ============================================================
cap.release()

df = pd.DataFrame(extracted_data)
output_path = 'extracted_champions.csv'
df.to_csv(output_path, index=False)

print(f"\nÖsszes feldolgozott másodperc: {len(df)}")
print(f"Adatok mentve: {output_path}")

FPS: 60.0

[00:30] --- Hősök azonosítása... ---

[DEBUG] Portrék megjelenítése ellenőrzéshez...
Nyomj meg bármilyen gombot az ablakok bezárásához és a folytatáshoz! (Ha a vágás rossz, zárd be a scriptet és írd át a koordinátákat)
Azonosítás eredménye:
  blue_top: Jayce_0
  blue_jungle: Qiyana_0
  blue_mid: Ryze_0
  blue_adc: Jhin_0
  blue_support: Alistar_0
  red_top: Gragas_0
  red_jungle: Naafiri_0
  red_mid: Orianna_0
  red_adc: Lucian_0
  red_support: Milio_0
------------------------------------

[00:30] Feldolgozás alatt...

Összes feldolgozott másodperc: 40
Adatok mentve: extracted_champions.csv
